# Fine-tuned ModernBERT — Supervised Token Classification

A supervised learning approach built on top of the `KRLabsOrg/lettucedect-large-modernbert-en-v1` checkpoint (a ModernBERT-large token classification backbone originally pre-trained on RAGTruth). 

The model is fine-tuned on our custom `combined_train.jsonl` dataset, which contains augmented Type 1 value-swaps, Type 2 overgenerations, and Type 3 missing-tool hallucinations, balanced with clean tool-calling interactions. This step adapts the token classifier specifically to the structural boundaries and API schemas of tool-calling dialogues, aiming to improve on the off-the-shelf baseline's precision.

In [1]:
!pip install --upgrade pip

!pip install torch --index-url https://download.pytorch.org/whl/cu121

!pip install "transformers>=4.48.0" "accelerate>=0.26.0" pandas numpy tqdm ipywidgets

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 13.5 MB/s eta 0:00:00 0:00:01
  Attempting uninstall: pip
    Found existing installation: pip 22.3.1
    Uninstalling pip-22.3.1:
      Successfully uninstalled pip-22.3.1
Looking in indexes: https://download.pytorch.org/whl/cu121
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 MB 53.0 MB/s  0:00:13:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 82.9 MB/s  0:00:00 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 81.0 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 102.4 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 83.3 MB/s  0:00:06:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 96.7 MB/s  0:00:03:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 101.4 MB/s  0:00:01a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 MB 102.7 MB/s  0:00:00eta 0:00:01
     ━━━━

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
import json
import torch
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from dataclasses import dataclass
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForTokenClassification, Trainer, TrainingArguments

os.makedirs("/app/results", exist_ok=True)
OUTPUT_DIR = "/app/checkpoints/modernbert_finetuned"
DATA_DIR = Path("/app/data")

TEST_FILES = {
    "Combined": DATA_DIR / "combined_test.jsonl",
    "Type 1 + clean": DATA_DIR / "type1_test_balanced.jsonl",
    "Type 2 + clean": DATA_DIR / "type2_test_balanced.jsonl",
    "Type 3 + clean": DATA_DIR / "type3_test_balanced.jsonl",
}

BASE_MODEL = "KRLabsOrg/lettucedect-large-modernbert-en-v1"
MAX_LENGTH = 2048
BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 1
EPOCHS = 3
LR = 1e-5
CLEAN_OVERSAMPLE = 4
SEED = 42

torch.manual_seed(SEED)

def read_jsonl(path):
    with open(path, "r", encoding="utf-8") as f: return [json.loads(line) for line in f if line.strip()]

train_samples, val_samples = read_jsonl(DATA_DIR / "combined_train.jsonl"), read_jsonl(DATA_DIR / "combined_val.jsonl")
clean = [s for s in train_samples if not s.get("hallucination_labels")]
hallu = [s for s in train_samples if s.get("hallucination_labels")]
train_samples = hallu + clean * CLEAN_OVERSAMPLE

print(f"Train: {len(train_samples)}, Val: {len(val_samples)}")

Train: 12395, Val: 200


In [3]:
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForTokenClassification.from_pretrained(BASE_MODEL, num_labels=2, ignore_mismatched_sizes=True)
model.to("cuda")

def encode_sample(sample, max_length):
    enc_ctx = tokenizer(sample.get("context", ""), add_special_tokens=False, truncation=False)
    enc_q   = tokenizer(sample.get("query", ""), add_special_tokens=False, truncation=False)
    enc_a   = tokenizer(sample.get("output", ""), add_special_tokens=False, return_offsets_mapping=True, truncation=False)
    
    ids = [tokenizer.cls_token_id] + enc_ctx["input_ids"] + [tokenizer.sep_token_id] + enc_q["input_ids"] + [tokenizer.sep_token_id]
    ans_start = len(ids)
    ids += enc_a["input_ids"] + [tokenizer.sep_token_id]
    
    if len(ids) > max_length:
        keep = max(0, len(enc_ctx["input_ids"]) - (len(ids) - max_length))
        ids = [tokenizer.cls_token_id] + enc_ctx["input_ids"][:keep] + [tokenizer.sep_token_id] + enc_q["input_ids"] + [tokenizer.sep_token_id]
        ans_start = len(ids)
        ids += enc_a["input_ids"] + [tokenizer.sep_token_id]
        
    labels = [-100] * len(ids)
    for span in sample.get("hallucination_labels", []):
        s, e = int(span["start"]), int(span["end"])
        for i, (a, b) in enumerate(enc_a["offset_mapping"]):
            if a != b and a < e and b > s: labels[ans_start + i] = 1
            elif labels[ans_start + i] != 1: labels[ans_start + i] = 0
            
    return {"input_ids": ids, "attention_mask": [1]*len(ids), "labels": labels}

class HallucinationDS(Dataset):
    def __init__(self, samples): self.samples = samples
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx): return encode_sample(self.samples[idx], MAX_LENGTH)

def collate(batch):
    ml = max(len(b["input_ids"]) for b in batch)
    pad = lambda seq, val: seq + [val] * (ml - len(seq))
    return {
        "input_ids": torch.tensor([pad(b["input_ids"], tokenizer.pad_token_id or 0) for b in batch], dtype=torch.long),
        "attention_mask": torch.tensor([pad(b["attention_mask"], 0) for b in batch], dtype=torch.long),
        "labels": torch.tensor([pad(b["labels"], -100) for b in batch], dtype=torch.long),
    }

trainer = Trainer(
    model=model, 
    args=TrainingArguments(
        output_dir=OUTPUT_DIR, per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
        learning_rate=LR, num_train_epochs=EPOCHS, eval_strategy="epoch", save_strategy="epoch", bf16=True, report_to="none", load_best_model_at_end=True
    ),
    train_dataset=HallucinationDS(train_samples), eval_dataset=HallucinationDS(val_samples), data_collator=collate
)
trainer.train()
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.58M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.58G [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/app/checkpoints/modernbert_finetuned/tokenizer_config.json',
 '/app/checkpoints/modernbert_finetuned/tokenizer.json')

In [7]:
def _char_set(spans): return {i for s in spans for i in range(int(s["start"]), int(s["end"]))}

def span_metrics(samples, pred_spans):
    micro_overlap = micro_pred = micro_gold = 0
    macro_f1 = []
    for sample, preds in zip(samples, pred_spans):
        gold = _char_set(sample.get("hallucination_labels", [])); pred = _char_set(preds)
        overlap = len(gold & pred)
        micro_overlap += overlap; micro_pred += len(pred); micro_gold += len(gold)
        if not pred and not gold: macro_f1.append(1.0); continue
        p = overlap / len(pred) if pred else 0.0; r = overlap / len(gold) if gold else 0.0
        macro_f1.append(2 * p * r / (p + r) if (p + r) > 0 else 0.0)
    p_mi = micro_overlap / micro_pred if micro_pred else 0.0
    r_mi = micro_overlap / micro_gold if micro_gold else 0.0
    f_mi = 2 * p_mi * r_mi / (p_mi + r_mi) if (p_mi + r_mi) > 0 else 0.0
    return {"P": p_mi, "R": r_mi, "F1": f_mi, "macro_F1": sum(macro_f1)/len(macro_f1) if macro_f1 else 0.0}

def example_metrics(samples, pred_spans):
    tp = fp = fn = 0
    for sample, preds in zip(samples, pred_spans):
        gold = 1 if sample.get("hallucination_labels") else 0; pred = 1 if preds else 0
        if gold and pred: tp += 1
        elif gold: fn += 1
        elif pred: fp += 1
    p = tp / (tp + fp) if (tp + fp) else 0.0; r = tp / (tp + fn) if (tp + fn) else 0.0
    return {"P": p, "R": r, "F1": 2 * p * r / (p + r) if (p + r) > 0 else 0.0}

def predict_spans(sample):
    enc_a = tokenizer(sample.get("output", ""), add_special_tokens=False, return_offsets_mapping=True)
    e = encode_sample(sample, MAX_LENGTH)
    with torch.no_grad():
        out = model(input_ids=torch.tensor([e["input_ids"]], device="cuda:0"), attention_mask=torch.tensor([e["attention_mask"]], device="cuda:0"))
    probs = torch.softmax(out.logits[0], dim=-1)[:, 1].cpu().tolist()
    ans_start = e["input_ids"].index(tokenizer.sep_token_id, e["input_ids"].index(tokenizer.sep_token_id)+1) + 1
    a_probs = probs[ans_start : ans_start+len(enc_a["input_ids"])]
    
    spans, cs, ce = [], None, None
    for (a, b), p in zip(enc_a["offset_mapping"], a_probs):
        if a == b: continue
        if p > 0.5:
            if cs is None: cs = a
            ce = b
        else:
            if cs is not None: spans.append({"start": cs, "end": ce}); cs = ce = None
    if cs is not None: spans.append({"start": cs, "end": ce})
    return spans

results = []
for name, path in TEST_FILES.items():
    if not path.exists(): continue
    samples = read_jsonl(path)
    preds = [predict_spans(s) for s in tqdm(samples, desc=name)]
    sm, em = span_metrics(samples, preds), example_metrics(samples, preds)
    results.append({
        "Split": name, "N": len(samples),
        "Span P": round(sm["P"], 3), "Span R": round(sm["R"], 3), "Span F1": round(sm["F1"], 3), "Span macro F1": round(sm["macro_F1"], 3),
        "Ex P": round(em["P"], 3), "Ex R": round(em["R"], 3), "Ex F1": round(em["F1"], 3)
    })

with open("/app/results/finetuned_modernbert_metrics.json", "w", encoding="utf-8") as f: json.dump(results, f, indent=4)
display(pd.DataFrame(results))

Combined:   0%|          | 0/599 [00:00<?, ?it/s]

Type 1 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

Type 2 + clean:   0%|          | 0/300 [00:00<?, ?it/s]

Type 3 + clean:   0%|          | 0/299 [00:00<?, ?it/s]

,Split,N,Span P,Span R,Span F1,Span macro F1,Ex P,Ex R,Ex F1
0,Combined,599,0.962,0.997,0.979,0.910,0.951,0.991,0.971
1,Type 1 + clean,300,0.636,0.949,0.762,0.826,0.864,0.973,0.915
2,Type 2 + clean,300,0.970,1.000,0.985,0.921,0.867,1.000,0.929
3,Type 3 + clean,299,0.963,1.000,0.981,0.921,0.866,1.000,0.928
